In [1]:
import os

import pandas as pd
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from esp_data.file_io.utils import make_fs
from esp_data.paths import AnyPath

In [3]:
import json

In [16]:
from beans_cfg import DATASET_JSONL_PATHS, Beans0SampleNoAudio, beans0_cfg

In [7]:
def read_jsonl(path: str | AnyPath) -> list[dict]:
    try:
        with open(path, "r") as f:
            data = json.load(f)
            annotation = data["annotation"]
    except (json.JSONDecodeError, KeyError):
        with open(path, "r") as f:
            annotation = [json.loads(line) for line in f]
    return annotation

In [51]:
B = AnyPath("gs://foundation-model-data/beans0/raw/metadata.csv")
B.exists()

True

In [52]:
B.unlink()
B.exists()

False

In [13]:
local_dir = AnyPath("./beans0/jsonl_files/")

In [9]:
fs = make_fs("gs://")
type(fs)

gcsfs.core.GCSFileSystem

In [33]:
# download
# for k, v in DATASET_JSONL_PATHS.items():
#    fs.get(v, local_dir / (k + ".jsonl"))

In [14]:
# load all
all_rows = []
for k in list(DATASET_JSONL_PATHS.keys()):
    rows = read_jsonl(local_dir / (k + ".jsonl"))
    all_rows.extend(rows)

In [15]:
len(all_rows)

76620

In [29]:
all_rows[40]

{'instruction': '{esc50-animal}',
 'path': '/home/davidrobinson/foundation-model-data/audio_16k/esc50/5-200334-B-1.wav',
 'label': 'rooster',
 'text': 'rooster'}

In [25]:
all_rows[-1]

{'prompt': "<Audio><AudioHere></Audio> Is there only one bird in the audio, or more? Reply with 'One' or 'More'.",
 'text': 'More',
 'path': '../audio/zebra_finch_elie/multiple_birds1.0/contact_WhiLbl0010LblRed0613WhiBlu3513WhiWhi1415_194.wav'}

In [19]:
from esp_data.utils import make_simple_logger

In [20]:
logger = make_simple_logger("beans0", add_file_handler=False)

In [45]:
def make_sample(
    row: dict,
    source_dataset: str,
    license: str,
    replace_16k: bool = True,
) -> dict:
    """Make a single BeansSample from a row in a component Beans0 dataset.

    Args:
        row (dict): The row from the component dataset.
        source_dataset (str): The name of the component dataset.
        license (str): The license for the component dataset.
        replace_16k (bool): Whether to replace 16k audio with original audio.

    Returns:
        dict: The Beans0Sample.
    """
    path_parts = AnyPath(row["path"]).parts

    ## VERY HACKY, issue is paths are local to David's VM
    animal_speak = False
    if "foundation-model-data" in path_parts:
        idx_go = path_parts.index("foundation-model-data")
    elif "animalspeak2" in path_parts:
        # look for "animalspeak2"
        idx_go = path_parts.index("animalspeak2")
        animal_speak = True
    elif "zebra_finch_elie" in path_parts:
        path_parts = ["foundation-model-data"] + list(path_parts[1:])
        idx_go = 0
    else:
        logger.error(f"Path {path_parts} not in expected format")
        return None

    path = "gs://" + "/".join(path_parts[idx_go:])

    if replace_16k and not animal_speak:
        # ASSUMPTION: original audio is in "audio" and 16k audio is in "audio_16k"
        path = path.replace("audio_16k", "audio")

    original_path = path
    # audio_file = GSAudioFile(path)

    # if not audio_file.exists:
    #     logger.error(f"Sample not found at {path}")
    #     return None

    # check if row has a license and recordist field
    metadata = {}
    if "license" in row:
        license = row["license"]
    if "recordist" in row:
        recordist = row["recordist"]
        metadata = {"recordist": recordist}
    if "url" in row:
        metadata["url"] = row["url"]

    # HACK, text is not present in all datasets, sometimes 'answer' is present
    if "text" not in row and "answer" in row:
        row["text"] = row["answer"]

    if "source" in row:
        metadata["source"] = row["source"]
        source_dataset = row["source"]
        if source_dataset == "iNaturalist":
            metadata["exclude_from_release"] = True

    file_name = os.path.basename(path)

    try:
        sample = Beans0SampleNoAudio(
            source_dataset=source_dataset,
            license=license,
            metadata=metadata,
            file_name=file_name,
            prompt=row.get("prompt", "None"),  ## HACK for esc50
            text=row.get("text", "None"),
            task=row.get("task", "None"),
        )
        sample = sample.to_dict()
        # remove derived_from and version
        sample.pop("derived_from", None)
        sample.pop("version", None)

        # copy the audio file to the output_dir
        # if not F.exists(output_dir / file_name):
        #    audio_file.copy_to(output_dir / file_name)

        return sample, original_path

    except Exception as e:
        logger.error(f"Pydantic Validation error {e}")
        return None, None

In [22]:
from tqdm.notebook import tqdm

In [46]:
samples = []
error = []
original_paths = []

In [47]:
for name in list(DATASET_JSONL_PATHS.keys()):
    print(f"\nProcessing dataset {name}")
    dataset_idx_in_cfg = [d["name"] for d in beans0_cfg.metadata["components"]].index(name)
    dataset_cfg = beans0_cfg.metadata["components"][dataset_idx_in_cfg]

    ds_rows = read_jsonl(local_dir / (name + ".jsonl"))

    for row in tqdm(ds_rows, total=len(ds_rows)):
        sample, original_path = make_sample(row, dataset_cfg["name"], dataset_cfg["license"], replace_16k=True)

        if sample is None:
            error.append(row)

        else:
            samples.append(sample)
            original_paths.append(original_path)


Processing dataset esc50


  0%|          | 0/96 [00:00<?, ?it/s]


Processing dataset watkins


  0%|          | 0/339 [00:00<?, ?it/s]


Processing dataset cbi


...

  0%|          | 0/951 [00:00<?, ?it/s]


Processing dataset captioning


  0%|          | 0/26468 [00:00<?, ?it/s]


Processing dataset zf-indiv


  0%|          | 0/1160 [00:00<?, ?it/s]

In [48]:
len(samples)

76620

In [49]:
samples_df = pd.DataFrame.from_records(samples)

In [53]:
samples_df.columns

Index(['source_dataset', 'metadata', 'id', 'created_at', 'license',
       'file_name', 'prompt', 'text', 'task'],
      dtype='object')

In [54]:
samples_df.to_csv("./metadata.csv", index=False)

In [55]:
samples_df.duplicated(subset=["source_dataset", "file_name", "prompt", "text", "task"]).sum()

np.int64(0)

In [56]:
len(original_paths)

76620

In [57]:
original_paths[-1]

'gs://foundation-model-data/audio/zebra_finch_elie/multiple_birds1.0/contact_WhiLbl0010LblRed0613WhiBlu3513WhiWhi1415_194.wav'

In [58]:
original_paths[0]

'gs://foundation-model-data/audio/esc50/5-103415-A-2.wav'

In [59]:
original_paths = pd.Series(data=original_paths, name="path")
original_paths.head(3)

0    gs://foundation-model-data/audio/esc50/5-10341...
1    gs://foundation-model-data/audio/esc50/5-10341...
2    gs://foundation-model-data/audio/esc50/5-10341...
Name: path, dtype: object

In [60]:
original_paths.to_csv("./original_paths.csv", index=False)